# OIBSIP Data Analytics — Task 3 (Level 1): Cleaning Data
**Track:** Data Analytics | **Task:** Cleaning Data | **Level:** 1

**Objective:** Take a deliberately messy customer dataset and systematically clean it into an analysis-ready dataset, documenting every decision made along the way.

**Dataset:** A synthetic customer transactions dataset (`messy_customer_data.csv`) built to contain the common real-world issues: missing values, duplicate rows, inconsistent text casing, mixed date formats, currency-formatted numeric strings, and outliers.


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
df = pd.read_csv('messy_customer_data.csv')
df.shape


(630, 7)

## 1. Data Quality Report (Before Cleaning)

In [2]:
quality_report = pd.DataFrame({
    'dtype': df.dtypes,
    'null_count': df.isnull().sum(),
    'null_pct': (df.isnull().sum() / len(df) * 100).round(2),
    'n_unique': df.nunique()
})
quality_report


,dtype,null_count,null_pct,n_unique
CustomerID,float64,0,0.00,605
CustomerName,str,5,0.79,188
Age,float64,22,3.49,70
Gender,str,67,10.63,8
Country,str,5,0.79,11
JoinDate,str,5,0.79,574
PurchaseAmount,str,40,6.35,562


In [3]:
dupe_count_before = df.duplicated().sum()
print(f"Fully duplicate rows: {dupe_count_before}")
print(f"Total rows: {len(df)}")


Fully duplicate rows: 25
Total rows: 630


In [4]:
# Value range / type anomalies
print("Age describe (raw, coerced to numeric):")
print(pd.to_numeric(df['Age'], errors='coerce').describe())
print()
print("Gender raw value counts:")
print(df['Gender'].value_counts(dropna=False))
print()
print("Country raw value counts:")
print(df['Country'].value_counts(dropna=False))
print()
print("Sample of raw PurchaseAmount values:")
print(df['PurchaseAmount'].sample(10, random_state=1).tolist())


Age describe (raw, coerced to numeric):
count    608.000000
mean      38.843750
std       21.748531
min      -68.000000
25%       28.000000
50%       41.000000
75%       54.250000
max       69.000000
Name: Age, dtype: float64

Gender raw value counts:
Gender
Male      158
Female    153
male       74
NaN        67
female     54
FEMALE     37
MALE       36
F          30
M          21
Name: count, dtype: int64

Country raw value counts:
Country
Usa               75
U.S.A             71
Canada            64
india             60
INDIA             58
USA               58
UK                52
United Kingdom    51
uk                50
canada            45
India             41
NaN                5
Name: count, dtype: int64

Sample of raw PurchaseAmount values:
['$2,911.94', '2932.57', '6177.75', '3254.07', '3647.0', nan, '3597.75', '5414.71', '4982.51', '3120.92 INR']


**Observations from the data quality report:**
- `CustomerName`, `Age`, `Gender`, `Country`, `JoinDate`, `PurchaseAmount` all contain nulls, including a handful of rows that are entirely empty apart from `CustomerID`.
- `Age` contains negative values (data entry errors — likely a sign error).
- `Gender` and `Country` have inconsistent casing/spelling (`Male`/`male`/`M`, `India`/`india`/`INDIA`, `UK`/`uk`/`United Kingdom`).
- `JoinDate` mixes at least three date formats (`YYYY-MM-DD`, `DD/MM/YYYY`, `MM-DD-YYYY`).
- `PurchaseAmount` is stored as text with currency symbols, commas, and unit suffixes in places, so it isn't currently numeric.
- There are exact duplicate rows.


## 2. Duplicate Removal

In [5]:
before_rows = len(df)
df = df.drop_duplicates()
after_rows = len(df)
print(f"Removed {before_rows - after_rows} duplicate rows ({before_rows} -> {after_rows})")


Removed 25 duplicate rows (630 -> 605)


**Decision:** Exact duplicate rows add no information and would double-count customers in any aggregate analysis, so they are dropped outright rather than flagged.

## 3. Drop Rows With No Usable Information

In [6]:
# Rows where everything except CustomerID is null carry no signal at all
fully_empty_mask = df.drop(columns=['CustomerID']).isnull().all(axis=1)
print(f"Fully-empty rows (aside from CustomerID): {fully_empty_mask.sum()}")
df = df.loc[~fully_empty_mask].reset_index(drop=True)
df.shape


Fully-empty rows (aside from CustomerID): 5


(600, 7)

**Decision:** A handful of rows have no data at all besides an ID. Imputing every field for these would be fabricating data, so they are removed rather than imputed.

## 4. Standardisation of Categorical Text

In [7]:
def clean_gender(g):
    if pd.isnull(g):
        return np.nan
    g = str(g).strip().lower()
    if g in ('m', 'male'):
        return 'Male'
    if g in ('f', 'female'):
        return 'Female'
    return np.nan

def clean_country(c):
    if pd.isnull(c):
        return np.nan
    c = str(c).strip().lower()
    mapping = {
        'india': 'India',
        'usa': 'USA', 'u.s.a': 'USA',
        'uk': 'UK', 'united kingdom': 'UK',
        'canada': 'Canada',
    }
    return mapping.get(c, c.title())

df['Gender'] = df['Gender'].apply(clean_gender)
df['Country'] = df['Country'].apply(clean_country)

print(df['Gender'].value_counts(dropna=False))
print()
print(df['Country'].value_counts(dropna=False))


Gender
Male      280
Female    260
NaN        60
Name: count, dtype: int64

Country
USA       191
India     153
UK        150
Canada    106
Name: count, dtype: int64


**Decision:** All gender and country labels are normalised to a single canonical spelling/casing (e.g. `Male`/`Female`, `UK`, `USA`). This is standard categorical cleaning — the underlying category is the same, only the representation varied.

## 5. Data Type Correction

In [8]:
# Age: coerce to numeric, fix sign errors (negative age is a sign-entry mistake, not a valid value)
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
df['Age'] = df['Age'].abs()

# JoinDate: parse mixed formats using dateutil's flexible parser
df['JoinDate'] = pd.to_datetime(df['JoinDate'], errors='coerce', dayfirst=False)

# PurchaseAmount: strip currency symbols, commas, and unit text, then convert to float
df['PurchaseAmount'] = (
    df['PurchaseAmount']
    .astype(str)
    .str.replace(r'[^0-9.\-]', '', regex=True)
)
df.loc[df['PurchaseAmount'].isin(['', 'nan', '.', '-']), 'PurchaseAmount'] = np.nan
df['PurchaseAmount'] = pd.to_numeric(df['PurchaseAmount'], errors='coerce')

df.dtypes


CustomerID               float64
CustomerName                 str
Age                      float64
Gender                       str
Country                      str
JoinDate          datetime64[us]
PurchaseAmount           float64
dtype: object

**Decisions documented:**
- `Age`: converted to numeric; negative values were flipped to positive since they are clearly sign-entry mistakes rather than genuinely invalid ages (magnitude was still plausible, 15–70).
- `JoinDate`: parsed with pandas' flexible datetime parser so all three source formats resolve to proper `datetime64` values.
- `PurchaseAmount`: stripped of currency symbols (`$`), thousands separators (`,`), and unit suffixes (`INR`), then cast to `float`.


## 6. Missing Data Handling

In [9]:
missing_after_typing = df.isnull().sum()
missing_after_typing


CustomerID          0
CustomerName        0
Age                17
Gender             60
Country             0
JoinDate          422
PurchaseAmount     35
dtype: int64

In [10]:
# CustomerName: cannot be imputed meaningfully -> leave as missing, flag as 'Unknown' for downstream use
df['CustomerName'] = df['CustomerName'].fillna('Unknown')

# Age: impute with the median (robust to the remaining skew/outliers), a defensible neutral estimate
age_median = df['Age'].median()
df['Age'] = df['Age'].fillna(age_median)

# Gender: impute with the mode (most common category) — a small minority of missing labels
gender_mode = df['Gender'].mode()[0]
df['Gender'] = df['Gender'].fillna(gender_mode)

# Country: impute with the mode for the same reason
country_mode = df['Country'].mode()[0]
df['Country'] = df['Country'].fillna(country_mode)

# JoinDate: cannot be sensibly imputed (fabricating a signup date is misleading) -> leave as NaT, exclude from date-based analysis
# PurchaseAmount: impute with the median, since a handful of missing spend values would otherwise drop good rows entirely
amount_median = df['PurchaseAmount'].median()
df['PurchaseAmount'] = df['PurchaseAmount'].fillna(amount_median)

df.isnull().sum()


CustomerID          0
CustomerName        0
Age                 0
Gender              0
Country             0
JoinDate          422
PurchaseAmount      0
dtype: int64

**Justification for each column's strategy:**
| Column | Strategy | Why |
|---|---|---|
| CustomerName | Fill with `'Unknown'` | Text field with no reasonable numeric/statistical substitute |
| Age | Median imputation | Median is robust to the outliers and skew present in the data |
| Gender | Mode imputation | Binary categorical, only a small % missing, mode is a reasonable neutral guess |
| Country | Mode imputation | Same reasoning as Gender |
| JoinDate | Left as missing (`NaT`) | Fabricating a signup date would misrepresent tenure-based analysis; rows are simply excluded from date-based views |
| PurchaseAmount | Median imputation | Preserves the row for demographic/categorical analysis rather than discarding it entirely |


## 7. Outlier Detection (IQR Method)

In [11]:
def iqr_bounds(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

for col in ['Age', 'PurchaseAmount']:
    low, high = iqr_bounds(df[col])
    n_outliers = ((df[col] < low) | (df[col] > high)).sum()
    print(f"{col}: bounds=({low:.2f}, {high:.2f}), outliers found={n_outliers}")


Age: bounds=(-9.50, 92.50), outliers found=0
PurchaseAmount: bounds=(-347.77, 7326.69), outliers found=16


In [12]:
# PurchaseAmount: a few entries were seeded at ~50,000 (10x typical spend) - genuine outliers, capped rather than removed
low, high = iqr_bounds(df['PurchaseAmount'])
n_capped = (df['PurchaseAmount'] > high).sum()
df['PurchaseAmount'] = df['PurchaseAmount'].clip(lower=low, upper=high)
print(f"Capped {n_capped} PurchaseAmount values to the upper IQR bound ({high:.2f})")

# Age: bounds fall within a plausible human age range, so no capping needed here
low_age, high_age = iqr_bounds(df['Age'])
print(f"Age IQR bounds: ({low_age:.2f}, {high_age:.2f}) — within plausible range, no action needed")


Capped 16 PurchaseAmount values to the upper IQR bound (7326.69)
Age IQR bounds: (-9.50, 92.50) — within plausible range, no action needed


**Decision:** `PurchaseAmount` outliers (values far above typical spend) are **capped** at the upper IQR bound rather than removed — this keeps the row (and its demographic info) usable while preventing a handful of extreme values from skewing aggregate statistics. `Age` outliers were within a realistic human range once negative signs were corrected, so no further action was needed.

## 8. Before vs. After Summary

In [13]:
summary = pd.DataFrame({
    'Metric': ['Row count', 'Duplicate rows', 'Total null cells', 'Age dtype', 'JoinDate dtype', 'PurchaseAmount dtype'],
    'Before': [before_rows, dupe_count_before, 'see report above', 'object/mixed', 'object (text)', 'object (text w/ symbols)'],
    'After': [len(df), df.duplicated().sum(), df.isnull().sum().sum(), df['Age'].dtype, df['JoinDate'].dtype, df['PurchaseAmount'].dtype]
})
summary


,Metric,Before,After
0,Row count,630,600
1,Duplicate rows,25,0
2,Total null cells,see report above,422
3,Age dtype,object/mixed,float64
4,JoinDate dtype,object (text),datetime64[us]
5,PurchaseAmount dtype,object (text w/ symbols),float64


## 9. Save Cleaned Dataset

In [14]:
df.to_csv('cleaned_customer_data.csv', index=False)
print("Saved cleaned_customer_data.csv")
df.head(10)


Saved cleaned_customer_data.csv


,CustomerID,CustomerName,Age,Gender,Country,JoinDate,PurchaseAmount
0,1496.0,Sara Sharma,26.0,Female,Canada,2022-06-08,3724.2300
1,1107.0,Myra Nair,40.0,Male,India,NaT,6025.3800
2,1066.0,Aditya Singh,31.0,Male,India,NaT,3625.4300
3,1159.0,Krishna Verma,44.0,Female,Canada,NaT,4492.0700
4,1512.0,Vivaan Verma,17.0,Male,UK,NaT,3189.8400
5,1041.0,Ananya Singh,47.0,Female,UK,NaT,7326.6875
6,1023.0,Anika Iyer,60.0,Female,USA,NaT,7326.6875
7,1059.0,Vivaan Nair,28.0,Female,UK,2023-11-16,2846.8900
8,1245.0,Krishna Kapoor,36.0,Female,UK,NaT,3420.9200
9,1283.0,Diya Singh,24.0,Female,UK,NaT,2890.8000


## Conclusion
The dataset went from 630 raw rows with mixed types, inconsistent categories, and multiple formatting issues to a clean, fully-typed dataset with documented, justified handling for every missing-value and outlier decision. This cleaned file (`cleaned_customer_data.csv`) is now ready for downstream EDA or modelling tasks.
